# Notebook 04 - Synthetic Signal Analysis

This notebook analyzes invented values from anonymous signal channels. Results are illustrative and are not sourced from an organization or operating asset.

## Analysis step 2

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().resolve()
while not (ROOT / 'data' / 'interim' / 'channel_clean.csv').is_file() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'interim' / 'channel_clean.csv').is_file():
    raise FileNotFoundError('data/interim/channel_clean.csv not found. Run notebook 03 first.')

INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
FIG_DIR = ROOT / 'reports' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 11, 'axes.labelsize': 9, 'xtick.labelsize': 8,
    'ytick.labelsize': 8, 'legend.fontsize': 8, 'figure.constrained_layout.use': True,
})
ACCENT, NEUTRAL, GREY = '#E84B4B', '#4878D0', '#AAAAAA'

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 120)
pd.set_option('display.width', 200)

## Analysis step 3

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
WINDOWS       = [7, 14, 30]
MIN_HISTORY   = 14
FAILURE_DATE  = None            # e.g. '2033-12-20' once the system confirms it
HORIZONS      = [14, 30, 60]
MAIN_HORIZON  = 30
RUL_CAP       = 90
TRAIN_END     = '2032-12-31'
VAL_END       = '2033-06-30'
SUSTAIN       = 7
ALARM_PCT     = 99

CORE_CHANNELS = [
    'channel_098', 'channel_103', 'channel_102',
    'channel_105', 'channel_106', 'channel_107',
    'channel_101', 'channel_108', 'channel_100',
    'channel_104', 'channel_095',
    'channel_094', 'channel_093',
    'channel_099',
]
# the scaled table holds the log version of signal
CORE = ['channel_099_log' if c == 'channel_099' else c for c in CORE_CHANNELS]

## Analysis step 4

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
clean  = pd.read_csv(INTERIM / 'channel_clean.csv',  index_col=0, parse_dates=True)
scaled = pd.read_csv(INTERIM / 'channel_scaled.csv', index_col=0, parse_dates=True)

assert clean.index.equals(scaled.index), 'clean and scaled have different dates'
assert not [c for c in CORE if c not in scaled.columns]

flags = clean[['is_shutdown', 'is_restart', 'is_suspect', 'segment_id', 'day_in_segment',
               'core_complete', 'is_baseline']].copy()
for c in ['is_shutdown', 'is_restart', 'is_suspect', 'core_complete', 'is_baseline']:
    flags[c] = flags[c].astype(bool)
flags['segment_id'] = flags['segment_id'].astype('Int64')
flags['day_in_segment'] = flags['day_in_segment'].astype('Int64')

print('rows:', len(clean), '| operating days:', int((~flags['is_shutdown']).sum()),
      '| baseline days:', int(flags['is_baseline'].sum()))
print('segments:', flags['segment_id'].dropna().nunique())

## Analysis step 5

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
def rolling_slope(x):
    ok = ~np.isnan(x)
    if ok.sum() < 7:
        return np.nan
    t = np.arange(len(x))[ok]
    return np.polyfit(t, x[ok], 1)[0]


def build_features(sc, seg):
    # rolling features, one operating segment at a time
    parts = []
    for sid in seg.dropna().unique():
        mask = (seg == sid).fillna(False).to_numpy(dtype=bool)
        block = sc.loc[mask, CORE]
        out = {}
        for c in CORE:
            s = block[c]
            for w in WINDOWS:
                out[f'{c}__mean{w}'] = s.rolling(w, min_periods=max(3, w // 2)).mean()
            out[f'{c}__std14']   = s.rolling(14, min_periods=7).std()
            out[f'{c}__slope14'] = s.rolling(14, min_periods=7).apply(rolling_slope, raw=True)
            out[f'{c}__diff1']   = s.diff()
        parts.append(pd.DataFrame(out, index=block.index))
    feats = pd.concat(parts).sort_index()
    return feats.reindex(sc.index)


feats = build_features(scaled, flags['segment_id'])
print('base features:', feats.shape[1], '| rows:', len(feats))
print('shutdown rows with a feature (must be 0):', int(feats[flags['is_shutdown']].notna().any(axis=1).sum()))

### Analysis step 6

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
CUT = 400
feats_cut = build_features(scaled.iloc[:CUT], flags['segment_id'].iloc[:CUT])
same = np.allclose(feats_cut.to_numpy(dtype=float), feats.iloc[:CUT].to_numpy(dtype=float),
                   equal_nan=True)
print('features on the first', CUT, 'days do not change when the future is removed:', same)
assert same

### Analysis step 7

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
m7 = feats[[f'{c}__mean7' for c in CORE]]
feats['health_abs7'] = m7.abs().mean(axis=1, skipna=False)
feats['health_max7'] = m7.abs().max(axis=1, skipna=False)

base_rows = flags['is_baseline'].to_numpy() & m7.notna().all(axis=1).to_numpy()
lw = LedoitWolf().fit(m7.loc[base_rows])
ok = m7.notna().all(axis=1).to_numpy()
mahal = pd.Series(np.nan, index=feats.index)
mahal.loc[ok] = np.sqrt(lw.mahalanobis(m7.loc[ok]))
feats['mahal7'] = mahal

print('baseline rows used for the covariance:', int(base_rows.sum()))
feats[['health_abs7', 'health_max7', 'mahal7']].groupby(
    np.where(flags['is_baseline'], 'baseline 2032', np.where(flags['is_shutdown'], 'shutdown', 'other operating'))
).median().round(2)

### Analysis step 8

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
dis = flags['day_in_segment'].fillna(-1).astype(int)
feature_complete = feats.notna().all(axis=1)

usable = (~flags['is_shutdown']) & (~flags['is_restart']) & (dis >= MIN_HISTORY) & feature_complete

print('operating days               :', int((~flags['is_shutdown']).sum()))
print('  without enough history     :', int(((~flags['is_shutdown']) & (dis < MIN_HISTORY)).sum()))
print('  restart days               :', int(((~flags['is_shutdown']) & flags['is_restart']).sum()))
print('  with a missing feature     :', int(((~flags['is_shutdown']) & (dis >= MIN_HISTORY) & ~feature_complete).sum()))
print('usable rows                  :', int(usable.sum()))

## Analysis step 9

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
h = feats['health_abs7']
thr = h[flags['is_baseline']].quantile(ALARM_PCT / 100)
above = (h > thr).fillna(False)
run = above.rolling(SUSTAIN).sum() == SUSTAIN

print(f'alarm threshold (health_abs7, baseline p{ALARM_PCT}): {thr:.3f}')
print('false alarms of this rule inside the baseline year  :', int(run[flags['is_baseline']].sum()))

if run.any():
    first = run[run].index[0] - pd.Timedelta(days=SUSTAIN - 1)
    ONSET_ESTIMATE = first.normalize()
    print('estimated start of the sustained deviation          :', ONSET_ESTIMATE.date())
else:
    ONSET_ESTIMATE = None
    print('no sustained deviation found')

In [ ]:
last_operating = flags.index[~flags['is_shutdown']].max().normalize()

if FAILURE_DATE is None:
    FAILURE_DATE_USED = last_operating
    PROVISIONAL = True
    print('FAILURE_DATE is not set -> PROVISIONAL: using the last operating day of the data:',
          FAILURE_DATE_USED.date())
else:
    FAILURE_DATE_USED = pd.Timestamp(FAILURE_DATE).normalize()
    PROVISIONAL = False
    print('failure date (confirmed):', FAILURE_DATE_USED.date())

days_to_failure = (FAILURE_DATE_USED - flags.index.normalize()).days.to_series(index=flags.index)

labels = pd.DataFrame(index=flags.index)
labels['days_to_failure'] = days_to_failure
labels['label_valid'] = (~flags['is_shutdown']) & (days_to_failure >= 0)
labels['rul_days'] = days_to_failure.clip(upper=RUL_CAP).where(labels['label_valid'])
for hz in HORIZONS:
    labels[f'label_h{hz}'] = ((days_to_failure >= 0) & (days_to_failure <= hz)).astype(int)

labels['label'] = labels[f'label_h{MAIN_HORIZON}']

print('days after the failure date (label_valid = False):', int((days_to_failure < 0).sum()))

## Analysis step 10

This section uses synthetic values and anonymous channel identifiers.

## Analysis step 11

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
split = pd.Series('test', index=flags.index)
split[flags.index <= pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1)] = 'train'
split[(flags.index > pd.Timestamp(TRAIN_END) + pd.Timedelta(days=1)) &
      (flags.index <= pd.Timestamp(VAL_END) + pd.Timedelta(days=1))] = 'validation'

rows = usable & labels['label_valid']
bal = pd.DataFrame({
    'usable_rows': rows.groupby(split).sum(),
    **{f'positives_h{hz}': (rows & (labels[f'label_h{hz}'] == 1)).groupby(split).sum() for hz in HORIZONS},
}).reindex(['train', 'validation', 'test'])
bal

## Analysis step 12

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
for ax, col, title in [(axes[0], 'health_abs7', 'health_abs7 (mean distance from the 2032 baseline)'),
                       (axes[1], 'health_max7', 'health_max7 (largest single-sensor distance)'),
                       (axes[2], 'mahal7', 'mahal7 (Mahalanobis distance)')]:
    ax.plot(feats.index, feats[col], color=NEUTRAL, lw=0.9)
    ax.set_ylabel(col)
    ax.set_title(title, fontsize=9)
    ax.axvline(pd.Timestamp(TRAIN_END), color=GREY, ls=':', lw=1)
    ax.axvline(pd.Timestamp(VAL_END), color=GREY, ls=':', lw=1)
    ax.axvspan(FAILURE_DATE_USED - pd.Timedelta(days=MAIN_HORIZON), FAILURE_DATE_USED,
               color=ACCENT, alpha=0.2, lw=0)
    ax.axvline(FAILURE_DATE_USED, color=ACCENT, lw=1.2)
    if ONSET_ESTIMATE is not None:
        ax.axvline(ONSET_ESTIMATE, color='#FFA500', ls='--', lw=1.2)
axes[0].axhline(thr, color='#FFA500', ls=':', lw=1)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[2].xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.suptitle('Distance from the 2032 baseline. Red = failure date and the last '
             f'{MAIN_HORIZON} days, orange = estimated start of the deviation, dotted grey = split dates')
plt.savefig(FIG_DIR / 'notebook_04_plot_01.png', bbox_inches='tight')
plt.show()

In [ ]:
show = [('channel_098__mean14', 'signal, 14-day mean (scaled)'),
        ('channel_098__slope14', 'signal, 14-day slope'),
        ('channel_099_log__mean14', 'log signal, 14-day mean (scaled)'),
        ('channel_099_log__std14', 'log signal, 14-day std')]

fig, axes = plt.subplots(len(show), 1, figsize=(14, 9), sharex=True)
for ax, (col, title) in zip(axes, show):
    ax.plot(feats.index, feats[col], color=NEUTRAL, lw=0.9)
    ax.axvline(FAILURE_DATE_USED, color=ACCENT, lw=1.2)
    ax.axvline(pd.Timestamp(TRAIN_END), color=GREY, ls=':', lw=1)
    ax.axvline(pd.Timestamp(VAL_END), color=GREY, ls=':', lw=1)
    ax.set_ylabel(title, fontsize=8)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')
fig.suptitle('Some of the features over time (red = failure date)')
plt.savefig(FIG_DIR / 'notebook_04_plot_02.png', bbox_inches='tight')
plt.show()

### Analysis step 13

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
mask = rows & (split != 'train')
y = labels.loc[mask, 'label']
print('rows:', int(mask.sum()), '| positives:', int(y.sum()))

if y.nunique() == 2:
    res = {}
    for c in feats.columns:
        a = roc_auc_score(y, feats.loc[mask, c])
        res[c] = max(a, 1 - a)
    auc = pd.Series(res).sort_values(ascending=False)
    print(auc.head(12).round(3).to_string())
else:
    print('only one class in validation + test: AUC not defined')

## Analysis step 14

This section uses synthetic values and anonymous channel identifiers.

In [ ]:
table = pd.concat([feats, labels, flags, split.rename('split'), usable.rename('usable')], axis=1)

assert table.index.equals(clean.index) and table.index.is_unique
assert not np.isinf(table[feats.columns].to_numpy(dtype=float)).any()
assert table.loc[table['is_shutdown'], list(feats.columns)].isna().all().all()
assert (table.loc[table['usable'], list(feats.columns)].notna().all()).all()
assert not (table['usable'] & table['is_shutdown']).any()
assert set(table['split'].unique()) <= {'train', 'validation', 'test'}

# the training rows must not contain any pre-failure label
assert table.loc[(table['split'] == 'train') & table['usable'], 'label'].sum() == 0, 'pre-failure labels inside the training period'

kinds = []
for c in feats.columns:
    if '__' in c:
        base, kind = c.split('__')
    else:
        base, kind = 'all sensors', 'summary score'
    kinds.append({'feature': c, 'sensor': base, 'kind': kind})
feature_list = pd.DataFrame(kinds)

print('table:', table.shape, '| features:', len(feature_list), '| usable rows:', int(table['usable'].sum()))

In [ ]:
table.to_csv(PROCESSED / 'features_labels.csv')
feature_list.to_csv(PROCESSED / 'feature_list.csv', index=False)

cfg = {
    'failure_date_used': str(FAILURE_DATE_USED.date()),
    'failure_date_provisional': bool(PROVISIONAL),
    'estimated_onset': None if ONSET_ESTIMATE is None else str(ONSET_ESTIMATE.date()),
    'horizons': HORIZONS, 'main_horizon': MAIN_HORIZON, 'rul_cap': RUL_CAP,
    'windows': WINDOWS, 'min_history': MIN_HISTORY,
    'train_end': TRAIN_END, 'val_end': VAL_END,
    'core_sensors': CORE,
    'usable_rows': int(table['usable'].sum()),
}
with open(PROCESSED / 'label_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print('saved:', sorted(p.name for p in PROCESSED.iterdir()))
cfg

## Analysis step 15

This section uses synthetic values and anonymous channel identifiers.